# Altered States — Jane Street, March 2014

[https://www.janestreet.com/puzzles/altered-states-index/](https://www.janestreet.com/puzzles/altered-states-index/)

## Goals

Fill a 5x5 grid with letters so that as many of the 50 U.S. states as possible can be spelled by
walking around the grid. This is a great hard version of an *optimization* puzzle so I can get practice with CP-SAT model.

## AI use disclaimer

I used AI to help me with Python syntax and debugging and in writing some of the more tedious helper functions.  I had it transcribe the rules as well.

## Rules

- Enter letters into a 5x5 grid.
- A state is **present** if its name (spaces removed, so `RHODEISLAND`, `NEWHAMPSHIRE`) can be
  spelled by a sequence of *King's moves*: each letter after the first must sit on a square
  horizontally, vertically, or diagonally adjacent to the previous letter's square.
- A state scores its own length: `OREGON` scores 6, `RHODEISLAND` scores 11.
- A state that can be spelled more than one way still scores once.
- Maximize the total score.

## The example

The 3x3 example grid

```
. I H
D A O
. W .
```

scores **13**, from `IDAHO` (5), `IOWA` (4), and `OHIO` (4). It misses HAWAII because you can't choose not to move, so no way to double the "i"

In [ ]:
from ortools.sat.python import cp_model

## The 50 states, and the grid size

In [4]:
STATE_NAMES = [
    "ALABAMA",
    "ALASKA",
    "ARIZONA",
    "ARKANSAS",
    "CALIFORNIA",
    "COLORADO",
    "CONNECTICUT",
    "DELAWARE",
    "FLORIDA",
    "GEORGIA",
    "HAWAII",
    "IDAHO",
    "ILLINOIS",
    "INDIANA",
    "IOWA",
    "KANSAS",
    "KENTUCKY",
    "LOUISIANA",
    "MAINE",
    "MARYLAND",
    "MASSACHUSETTS",
    "MICHIGAN",
    "MINNESOTA",
    "MISSISSIPPI",
    "MISSOURI",
    "MONTANA",
    "NEBRASKA",
    "NEVADA",
    "NEWHAMPSHIRE",
    "NEWJERSEY",
    "NEWMEXICO",
    "NEWYORK",
    "NORTHCAROLINA",
    "NORTHDAKOTA",
    "OHIO",
    "OKLAHOMA",
    "OREGON",
    "PENNSYLVANIA",
    "RHODEISLAND",
    "SOUTHCAROLINA",
    "SOUTHDAKOTA",
    "TENNESSEE",
    "TEXAS",
    "UTAH",
    "VERMONT",
    "VIRGINIA",
    "WASHINGTON",
    "WESTVIRGINIA",
    "WISCONSIN",
    "WYOMING",
]

GRID_SIZE = 5

# Printed in a grid for a square that was deliberately left without a letter.
BLANK = "."

### Useful helper functions

Next, we need to implement a series of useful helper functions.  We can test those functions.

In [ ]:
def letters_used_by(states):
    """Return the sorted list of distinct letters appearing in the given state names."""
    all_letters = "".join(states)
    return sorted(set(all_letters))


def all_cells(size):
    """Return every (row, column) pair of a square grid, reading across then down."""
    cells = []
    for row in range(size):
        for column in range(size):
            cells.append((row, column))
    return cells


def king_neighbors(row, column, size):
    """Return the cells a King can move to from (row, column).

    The eight surrounding cells, minus any that fall off the board. The starting cell itself is
    never included
    """
    neighbors = []
    for row_step in (-1, 0, 1):
        for column_step in (-1, 0, 1):
            is_staying_put = row_step == 0 and column_step == 0
            if is_staying_put:
                continue
            neighbor_row = row + row_step
            neighbor_column = column + column_step
            row_on_board = 0 <= neighbor_row < size
            column_on_board = 0 <= neighbor_column < size
            if row_on_board and column_on_board:
                neighbors.append((neighbor_row, neighbor_column))
    return neighbors

## Building the scoring function

We first need to build a function that takes as input a grid of letters, and outputs a score. This will effectively give our solver the thing to optimize for -- the score. 

Our generall strategy for this will be to, one by one, search for each state using breadth-first sweep over the word.  For speed implications, we only need to keep track of the ending square of a given search -- how we get there does not matter and there is no point in keeping track of multiple pathways of the same letters if the ending square is the same.  This will save us tons of valuable time.  

In [ ]:
def word_in_grid(grid, word):
    size = len(grid)
    frontier = []
    for row, column in all_cells(size):
        if grid[row][column] == word[0]:
            frontier.append((row, column))

    for next_letter in word[1:]:
        extended_frontier = []
        for cell in frontier:
            row, column = cell
            for neighbor in king_neighbors(row, column, size):
                neighbor_row, neighbor_column = neighbor
                if grid[neighbor_row][neighbor_column] == next_letter:
                    extended_frontier.append(neighbor)
        frontier = extended_frontier
        if len(frontier) == 0:
            return False

    return True


def find_spelling_path(grid, word):
    """Return one King's-move path spelling word, as a list of cells, or None if there is none."""
    size = len(grid)

    # A walk of length one can start on any square holding the word's first letter.
    frontier = {}
    for row, column in all_cells(size):
        if grid[row][column] == word[0]:  ## we have a starting letter
            frontier[(row, column)] = [(row, column)]

    # word[1:] is a slice: every letter after the first one. The first letter is not an
    # extension of anything -- it seeded the frontier above -- so the loop starts at index 1.
    for next_letter in word[1:]:
        extended_frontier = {}
        for cell, path_so_far in frontier.items():
            row, column = cell
            for neighbor in king_neighbors(row, column, size):
                neighbor_row, neighbor_column = neighbor
                letter_matches = grid[neighbor_row][neighbor_column] == next_letter
                # One path per ending square is enough; the first one found is as good as any.
                already_reached = neighbor in extended_frontier
                if letter_matches and not already_reached:
                    extended_frontier[neighbor] = path_so_far + [neighbor]
        frontier = extended_frontier
        if len(frontier) == 0:
            return None

    # Any surviving entry is a complete, legal spelling of the word.
    first_path = list(frontier.values())[0]
    return first_path


def states_in_grid(grid, states=STATE_NAMES):
    """Return {state: one path spelling it} for every state present in the grid."""
    found = {}
    for state in states:
        path = find_spelling_path(grid, state)
        found[state] = path
    return found


def score_grid(grid, states=STATE_NAMES):
    """Return the puzzle score of a grid: the total length of the distinct states it contains."""
    total = 0
    for state in states_in_grid(grid, states):
        total += len(state)
    return total

In [8]:
def show_grid(grid):
    """Print a grid as the puzzle asks for it: rows reading across."""
    for row in grid:
        print(" ".join(row))


def show_score_report(grid, states=STATE_NAMES):
    """Print the grid, the states it contains with their paths, and the total score."""
    show_grid(grid)
    print()
    found = states_in_grid(grid, states)
    for state in sorted(found, key=len, reverse=True):
        path_text = " -> ".join(f"({row},{column})" for (row, column) in found[state])
        print(f"{len(state):3d}  {state:<14} {path_text}")
    print()
    print("total score:", score_grid(grid, states), "from", len(found), "states")

## Step 2: validate the checker against the published example

The puzzle hands us one scored instance. If the checker disagrees with it, everything built on
top of it is worthless, so this cell asserts rather than prints.

In [9]:
EXAMPLE_GRID = [
    [BLANK, "I", "H"],
    ["D", "A", "O"],
    [BLANK, "W", BLANK],
]

example_states = states_in_grid(EXAMPLE_GRID)
assert sorted(example_states) == ["IDAHO", "IOWA", "OHIO"], sorted(example_states)
assert score_grid(EXAMPLE_GRID) == 13

# OHIO must be found even though it starts and ends on the same square: reuse is legal.
assert find_spelling_path(EXAMPLE_GRID, "OHIO") is not None

# HAWAII must not be found: reaching the second I would mean staying put.
assert find_spelling_path(EXAMPLE_GRID, "HAWAII") is None

show_score_report(EXAMPLE_GRID)

. I H
D A O
. W .

  5  IDAHO          (0,1) -> (1,0) -> (1,1) -> (0,2) -> (1,2)
  4  IOWA           (0,1) -> (1,2) -> (2,1) -> (1,1)
  4  OHIO           (1,2) -> (0,2) -> (0,1) -> (1,2)

total score: 13 from 3 states


## Step 3: the model

**Cell variables.** `cell_holds[(row, column, letter)]` is a Bool that is true when that square
carries that letter, with exactly one true per square. Booleans rather than a single integer per
cell because every constraint below asks "does this square hold *this* letter", which a Bool
answers directly and an integer only answers through an extra reified comparison.

Leaving a square blank is allowed by the rules but can never help — adding a letter to an empty
square only creates new walks, it destroys none — so every square is required to hold a letter.

**Path variables.** The hard part is expressing "this word can be walked". The frontier idea from
`find_spelling_path` translates directly into variables: for each state, and each step of that
state's spelling, and each square, a Bool

> `spelled_through[step][cell]` — *some legal walk spells the word's first `step + 1` letters and
> ends on `cell`*

with two requirements:

- the square must actually hold the step's letter, and
- for every step after the first, some King's-move neighbor must carry the previous step.

`present[state]` then requires that some square carries the final step.

**One-directional implications are enough.** Each rule above is written as `variable => condition`
and never the reverse, so the solver can always set these Bools to false, but can only set one to
true when a genuine walk backs it up. It is free to *understate* a grid, but it cannot invent a
state that isn't there. Since the objective pays for every true `present[state]`, the solver has
every incentive to set them true wherever it legitimately can — a maximization objective supplies
the missing direction for free, and skipping the reverse implications keeps the model much
smaller. The consequence for us: the objective value is a **lower bound** on the grid's real
score, so the final answer always comes from re-running `score_grid` on the returned grid.

In [ ]:
def build_model(size=GRID_SIZE, states=STATE_NAMES):
    """Build the Altered States model.

    Returns (model, cell_holds, state_is_present), where cell_holds[(row, column, letter)] is
    true when that square carries that letter, and state_is_present[state] is the Bool the
    objective pays for.
    """
    model = cp_model.CpModel()
    cells = all_cells(size)
    alphabet = letters_used_by(states)

    cell_holds = {}
    for row, column in cells:
        for letter in alphabet:
            cell_holds[(row, column, letter)] = model.new_bool_var(
                f"cell_{row}{column}_holds_{letter}"
            )
        letters_here = [cell_holds[(row, column, letter)] for letter in alphabet]
        # Exactly one, not at most one: a blank square never helps, so blanks are ruled out.
        model.add_exactly_one(letters_here)

    state_is_present = {}
    for state in states:
        # spelled_through[step][cell]: a legal walk spells state[:step + 1] and ends on cell.
        spelled_through = []
        for step, letter in enumerate(state):
            layer = {}
            for row, column in cells:
                reached = model.new_bool_var(f"{state}_step{step}_at_{row}{column}")

                # Whatever the walk did earlier, this square has to hold this step's letter.
                model.add_implication(reached, cell_holds[(row, column, letter)])

                is_first_letter = step == 0
                if not is_first_letter:
                    # The previous letter has to sit on a square a King could have come from.
                    # king_neighbors excludes this square itself, which is what stops a walk
                    # from satisfying a repeated letter by standing still (the HAWAII rule).
                    arrivals = [
                        spelled_through[step - 1][neighbor]
                        for neighbor in king_neighbors(row, column, size)
                    ]
                    model.add_bool_or(arrivals).only_enforce_if(reached)

                layer[(row, column)] = reached
            spelled_through.append(layer)

        present = model.new_bool_var(f"present_{state}")
        final_step = spelled_through[-1]
        finishing_squares = [final_step[cell] for cell in cells]
        model.add_bool_or(finishing_squares).only_enforce_if(present)
        state_is_present[state] = present

    return model, cell_holds, state_is_present


def add_score_objective(model, state_is_present):
    """Maximize the puzzle score, and return the score expression for later reuse."""
    score = sum(len(state) * present for (state, present) in state_is_present.items())
    model.maximize(score)
    return score

## Step 4: optional extras

Two things worth having as separate switches, so their effect on runtime and on the optimum can
be measured rather than assumed.

**Symmetry breaking.** Rotating or reflecting a grid does not change its score, so every solution
belongs to a family of up to 8 identical-scoring twins that the solver would otherwise explore
separately. Comparing letters needs an ordering, so this introduces one integer per square
holding the letter's index in the alphabet. Requiring the top-left corner to be the smallest
corner rules out the three rotations; requiring `(0,1) <= (1,0)` then rules out the reflection
across the main diagonal, which is the one symmetry that fixes the top-left corner. Every family
keeps at least one member, so the optimum cannot move.

**Implied constraints.** If a state is present, each of its letters must appear somewhere in the
grid. That follows from the path variables already, so it adds nothing to what the model *means*
— but it is a fact the solver can check immediately instead of deriving through a chain of path
variables, which can prune whole branches early. Per the method: add it, then confirm the
objective did not move.

In [ ]:
def add_symmetry_breaking(model, cell_holds, size, alphabet):
    """Rule out rotated and reflected copies of a grid. Returns the per-cell letter-index vars."""
    index_of_letter = {}
    for index, letter in enumerate(alphabet):
        index_of_letter[letter] = index

    letter_index = {}
    for row, column in all_cells(size):
        index_here = model.new_int_var(
            0, len(alphabet) - 1, f"letter_index_{row}{column}"
        )
        for letter in alphabet:
            model.add(index_here == index_of_letter[letter]).only_enforce_if(
                cell_holds[(row, column, letter)]
            )
        letter_index[(row, column)] = index_here

    last = size - 1
    other_corners = [(0, last), (last, 0), (last, last)]
    for corner in other_corners:
        # Of the four corners, the top-left one holds the alphabetically smallest letter.
        # Some rotation of any grid satisfies this, so no score is lost.
        model.add(letter_index[(0, 0)] <= letter_index[corner])

    # The reflection across the main diagonal is the only symmetry that leaves the corner
    # ordering untouched, so it needs its own tie-break.
    model.add(letter_index[(0, 1)] <= letter_index[(1, 0)])

    return letter_index


def add_implied_letter_constraints(model, cell_holds, size, state_is_present):
    """Require every letter of a present state to appear somewhere in the grid."""
    for state, present in state_is_present.items():
        for letter in sorted(set(state)):
            squares_with_letter = [
                cell_holds[(row, column, letter)] for (row, column) in all_cells(size)
            ]
            model.add_bool_or(squares_with_letter).only_enforce_if(present)

## Step 5: reading a solution back out

`grid_from_solver` turns the Bools back into a printable grid. `states_claimed_by_solver` reports
what the model thinks it earned — which, per the one-directional encoding above, may be a subset
of what `states_in_grid` finds in the same grid. Comparing the two is the check that matters.

In [ ]:
def grid_from_solver(solver, cell_holds, size):
    """Return the solved grid as a list of rows of single-letter strings."""
    grid = []
    for row in range(size):
        grid.append([BLANK] * size)
    for (row, column, letter), holds_it in cell_holds.items():
        if solver.value(holds_it) == 1:
            grid[row][column] = letter
    return grid


def states_claimed_by_solver(solver, state_is_present):
    """Return the sorted states the model marked present."""
    claimed = []
    for state, present in state_is_present.items():
        if solver.value(present) == 1:
            claimed.append(state)
    return sorted(claimed)


def report_solution(
    solver, cell_holds, state_is_present, size=GRID_SIZE, states=STATE_NAMES
):
    """Print the solved grid and its independently re-checked score."""
    grid = grid_from_solver(solver, cell_holds, size)
    claimed = states_claimed_by_solver(solver, state_is_present)
    actually_present = sorted(states_in_grid(grid, states))

    show_score_report(grid, states)
    print()
    print("states the model claimed:", len(claimed))
    print("states the checker found:", len(actually_present))
    missed_by_model = sorted(set(actually_present) - set(claimed))
    if len(missed_by_model) > 0:
        # Expected and harmless: the objective had no reason to flag these, but they still score.
        print("present but unclaimed (free points):", missed_by_model)
    invented_by_model = sorted(set(claimed) - set(actually_present))
    # This one is never acceptable. It would mean the path encoding is wrong.
    assert len(invented_by_model) == 0, invented_by_model
    return grid

## Step 6: validate the encoding on a small instance

Before turning the solver loose on the real 5x5, run it on the example's own 3x3 board with only
the example's three states. The optimum there should be 13, matching the published example, and
the model's claim should agree exactly with the plain-Python checker. This is what makes a later
5x5 answer trustworthy.

In [ ]:
EXAMPLE_STATES = ["IDAHO", "IOWA", "OHIO"]

small_model, small_cell_holds, small_present = build_model(3, EXAMPLE_STATES)
add_score_objective(small_model, small_present)

small_solver = cp_model.CpSolver()
small_solver.parameters.max_time_in_seconds = 30
small_status = small_solver.solve(small_model)

assert small_status == cp_model.OPTIMAL
assert small_solver.objective_value == 13

small_grid = report_solution(
    small_solver, small_cell_holds, small_present, 3, EXAMPLE_STATES
)
assert score_grid(small_grid, EXAMPLE_STATES) == 13

## Step 7: solve the real grid

Not run to completion yet — this is the scaffolding. Things to try, roughly in order:

1. Solve with a time limit and no extras, to get a baseline score and see how fast the bound moves.
2. Add symmetry breaking, then the implied constraints, and confirm the best objective does not
   move while (hopefully) the runtime drops.
3. If proving optimality is out of reach, keep the best incumbent and improve it with
   `add_hint` from a good grid found so far.
4. Whatever comes out, re-score it with `score_grid` and submit the grid rows reading across,
   the total, and the list of states.

In [ ]:
model, cell_holds, state_is_present = build_model()
score = add_score_objective(model, state_is_present)

# Both are optional; comment them out to measure what they are worth.
add_symmetry_breaking(model, cell_holds, GRID_SIZE, letters_used_by(STATE_NAMES))
add_implied_letter_constraints(model, cell_holds, GRID_SIZE, state_is_present)

solver = cp_model.CpSolver()
solver.parameters.max_time_in_seconds = 60.0
solver.parameters.num_workers = 8
solver.parameters.log_search_progress = True

status = solver.solve(model)
print(solver.status_name(status))

In [ ]:
best_grid = report_solution(solver, cell_holds, state_is_present)